# CAZ Interactive Demo

*Run an LLM on your own concept pairs and watch CAZ emerge*

This notebook loads a language model and computes a full CAZ analysis on any concept you choose. Provide 10–20 contrastive sentence pairs and the notebook will show:
- Where in the model the concept crystallises (separation peak)
- Whether allocation is multimodal (two distinct assembly events)
- How your concept's peak depth compares to the 17 concepts from the paper

**Hardware requirements by model tier:**

| Tier | Models | VRAM | Notes |
|------|--------|------|-------|
| 1 — small | Qwen2.5-1.5B, GPT2-XL | ≤ 4 GB or CPU | **Default for ≤ 4 GB** — no quantization needed |
| 2 — medium | Qwen2.5-3B, Phi-2 | 4–8 GB | 4-bit quantized |
| 3 — paper | Qwen2.5-7B *(paper model)* | 8–16 GB | 4-bit, Colab T4 / Pro |
| 4 — large | Qwen2.5-14B, Llama-3.1-70B | 24+ GB | 4-bit, A100 / multi-GPU |

Your GPU is detected automatically and the best-fit model is pre-selected. **See § 1 to override.**

**[github.com/jamesrahenry/Rosetta](https://github.com/jamesrahenry/Rosetta)** · Henry (2026a)

In [1]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    _pip("rosetta_tools>=1.3.1")
except subprocess.CalledProcessError:
    _pip("rosetta_tools @ git+https://github.com/jamesrahenry/Rosetta_Tools.git@v1.3.1")

_pip("torch", "transformers", "accelerate", "huggingface_hub")

# bitsandbytes enables 4-bit quantization for Tier 2–4 models.
# Skip silently if installation fails (not needed for Tier 1 / CPU).
try:
    _pip("bitsandbytes")
except subprocess.CalledProcessError:
    print("bitsandbytes not installed — Tier 1 models (≤ 1.5 B) will still work fine.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download

from rosetta_tools.extraction import extract_contrastive_activations
from rosetta_tools.caz import compute_layer_metrics, find_caz_regions

print("Imports OK.")

Device: cuda
GPU: NVIDIA RTX 500 Ada Generation Laptop GPU  |  VRAM: 4.3 GB


In [3]:
from rosetta_tools.viz_style import (
    concept_color, CONCEPT_COLORS, FAMILY_COLORS, THEME, apply_theme,
)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "figure.facecolor": "white",
})
print("Style loaded.")

Style loaded.


In [ ]:
# ── § 1  Model selection ─────────────────────────────────────────────────────
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name}  |  VRAM: {vram_gb:.1f} GB")
else:
    vram_gb = 0
    print("No GPU detected — will run on CPU.")

# ─── Auto-recommendation ──────────────────────────────────────────────────────
if vram_gb >= 16:
    _rec = "Qwen/Qwen2.5-7B"
    _rec_tier = "Tier 3 — paper model; section 5 baseline matches exactly"
elif vram_gb >= 6:
    _rec = "Qwen/Qwen2.5-3B"
    _rec_tier = "Tier 2 — solid demo quality; 4-bit quantized"
elif vram_gb >= 2:
    _rec = "Qwen/Qwen2.5-1.5B"
    _rec_tier = "Tier 1 — fits in ≤ 4 GB VRAM; no quantization needed"
else:
    _rec = "openai-community/gpt2-xl"
    _rec_tier = "CPU fallback — no GPU or < 2 GB VRAM"

print(f"\nRecommended: {_rec}")
print(f"  {_rec_tier}")

# ─── Override here ────────────────────────────────────────────────────────────
# Uncomment any line below to use a different model, then re-run this cell.
#
# TIER 1 — CPU / ≤ 4 GB VRAM  (no quantization; bitsandbytes not required)
#   MODEL_ID = "Qwen/Qwen2.5-1.5B"           # 28L, 1536-dim, ~3 GB fp16  ← default ≤ 4 GB
#   MODEL_ID = "openai-community/gpt2-xl"     # 48L, 1600-dim, ~3 GB fp32  (CPU-friendly)
#   MODEL_ID = "openai-community/gpt2-large"  # 36L, 1280-dim, ~1.5 GB fp32
#   MODEL_ID = "EleutherAI/pythia-160m"       # 12L,  768-dim, < 1 GB      (fastest)
#   MODEL_ID = "Qwen/Qwen2.5-0.5B"           # 24L,  896-dim, ~1 GB
#
# TIER 2 — 4–8 GB VRAM  (4-bit quantized; requires bitsandbytes)
#   MODEL_ID = "Qwen/Qwen2.5-3B"             # 36L, 2048-dim, ~6 GB fp16 / ~2 GB 4-bit
#   MODEL_ID = "microsoft/phi-2"              # 32L, 2560-dim, ~5 GB fp16 / ~1.5 GB 4-bit
#   MODEL_ID = "EleutherAI/pythia-2.8b"      # 32L, 2560-dim, ~6 GB fp16 / ~2 GB 4-bit
#
# TIER 3 — 8–16 GB VRAM  (paper model; 4-bit on Colab T4 / Pro)   baseline ✓
#   MODEL_ID = "Qwen/Qwen2.5-7B"             # 28L, 3584-dim  ← PAPER MODEL
#   MODEL_ID = "meta-llama/Llama-3.1-8B"     # 32L, 4096-dim
#   MODEL_ID = "EleutherAI/pythia-6.9b"      # 32L, 4096-dim
#   MODEL_ID = "mistralai/Mistral-7B-v0.3"   # 32L, 4096-dim
#
# TIER 4 — 24+ GB VRAM  (A100 / multi-GPU)   baseline ✓
#   MODEL_ID = "Qwen/Qwen2.5-14B"            # 48L, 5120-dim
#   MODEL_ID = "Qwen/Qwen2.5-72B"            # 80L, 8192-dim
#   MODEL_ID = "meta-llama/Llama-3.1-70B"    # 80L, 8192-dim
# ─────────────────────────────────────────────────────────────────────────────

MODEL_ID = _rec  # ← change this line to override, e.g.  MODEL_ID = "Qwen/Qwen2.5-7B"

print(f"\nSelected: {MODEL_ID}")
print("Run § 2 to load the model.")

## 2. Load the model

The cell below loads the model selected in § 1. Loading strategy is chosen automatically:

- **Tier 1 (≤ 1.5 B):** fp16 on GPU, fp32 on CPU — `bitsandbytes` not required.
- **Tier 2–4 (3 B+):** 4-bit quantization via `bitsandbytes` to fit in GPU memory.

CAZ only needs `output_hidden_states=True`, which works identically on quantized models — quantization affects compute precision, not the geometric structure of the residual stream we measure.

*First download may take a few minutes. Subsequent runs use the cached weights.*

In [4]:
# Models that load in fp16/fp32 without 4-bit quantization
_TIER1 = {
    "openai-community/gpt2", "openai-community/gpt2-medium",
    "openai-community/gpt2-large", "openai-community/gpt2-xl",
    "EleutherAI/pythia-70m", "EleutherAI/pythia-160m", "EleutherAI/pythia-410m",
    "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B",
}

_use_quant = MODEL_ID not in _TIER1 and device == "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if _use_quant:
    from transformers import BitsAndBytesConfig
    _bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=_bnb, device_map="auto",
    )
else:
    _dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=_dtype,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to(device)

model.eval()

n_layers = model.config.num_hidden_layers
print(f"Loaded: {MODEL_ID}")
print(f"  {n_layers} layers  |  {model.config.hidden_size}-dim hidden")
print(f"  Loading mode: {'4-bit quantized' if _use_quant else ('fp16' if device == 'cuda' else 'fp32 (CPU)')}")
if device == "cuda":
    used_gb = torch.cuda.memory_allocated(0) / 1e9
    print(f"  VRAM in use: {used_gb:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

## 3. Define your concept

Edit the cell below:
- Set `CONCEPT_NAME` to a label for your concept
- Fill `pos_texts` with sentences that **clearly express** the concept
- Fill `neg_texts` with topically similar sentences that **lack** the concept

The contrast matters more than the size: 10–15 well-matched pairs work better than 30 loosely-matched ones. Keep `pos_texts` and `neg_texts` the same length.

The example below uses **causation** (from Paper 4). Swap it for anything you want to probe.

In [ ]:
# ── YOUR CONCEPT ─────────────────────────────────────────────────────────────
CONCEPT_NAME = "causation"

pos_texts = [
    "The bridge collapsed because the support beams had corroded.",
    "Scientists confirmed that the compound caused the cellular damage.",
    "Heavy rainfall led to widespread flooding across the valley.",
    "The fire spread rapidly due to the exceptionally dry conditions.",
    "Poor early nutrition resulted in slower cognitive development.",
    "The new austerity policy caused unemployment to rise sharply.",
    "Chronic stress triggered the patient's recurring migraines.",
    "The accident occurred as a direct result of driver fatigue.",
    "Her late arrival caused the entire meeting to be delayed.",
    "Prolonged exposure to the chemical produced severe respiratory burns.",
    "The burst pipe caused extensive water damage to the lower floors.",
    "Sleep deprivation led to serious errors in the team's judgement.",
]

neg_texts = [
    "The bridge collapsed, and the support beams were found to be corroded.",
    "Scientists observed the compound alongside the cellular damage.",
    "There was heavy rainfall, and the valley experienced widespread flooding.",
    "The conditions were exceptionally dry. The fire spread rapidly.",
    "The child had poor early nutrition and slower cognitive development.",
    "The new austerity policy was introduced. Unemployment rose sharply.",
    "The patient had chronic stress and recurring migraines.",
    "The driver was fatigued. The accident happened shortly after.",
    "She arrived late. The meeting was delayed.",
    "There was prolonged chemical exposure. Severe respiratory burns were present.",
    "The pipe burst. Water damage was found throughout the lower floors.",
    "The team members were sleep-deprived and made several serious errors.",
]
# ─────────────────────────────────────────────────────────────────────────────

assert len(pos_texts) == len(neg_texts), "pos and neg must be the same length"
print(f"Concept: {CONCEPT_NAME!r} — {len(pos_texts)} pairs")

## 4. Extract activations and compute CAZ

This runs all sentences through every layer of the model and computes $S(l)$, $C(l)$, $v(l)$ at each layer. With 12 pairs: ~5–15 s on a small Tier 1 model, ~20–40 s on a T4 with a 7B model.

In [ ]:
print("Extracting activations across all layers...")
layer_acts = extract_contrastive_activations(
    model, tokenizer, pos_texts, neg_texts,
    device=device, batch_size=4,
)

print("Computing CAZ metrics...")
metrics = compute_layer_metrics(layer_acts)
profile = find_caz_regions(metrics)
depth_pct = [m.layer / n_layers * 100 for m in metrics]

print(f"\nCAZ profile for '{CONCEPT_NAME}':")
print(f"  Allocation regions: {profile.n_regions}")
print(f"  Multimodal: {profile.is_multimodal}")
for i, region in enumerate(profile.regions):
    print(f"  Region {i+1}: peak layer {region.peak}  ({region.depth_pct:.1f}% depth)  "
          f"S={region.peak_separation:.3f}  C={region.peak_coherence:.3f}")

## 5. Visualise the allocation profile

The vertical dashed lines mark detected CAZ peaks. If there are two, the model is assembling the concept at two distinct depths — a shallow structural representation and a deeper semantic one.

In [ ]:
SIG_COLORS = {"separation": "#1565C0", "coherence": "#2E7D32", "velocity": "#E65100"}

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
fig.patch.set_facecolor("white")
fig.suptitle(f"CAZ profile — '{CONCEPT_NAME}'  ({MODEL_ID.split('/')[-1]})",
             fontsize=13, fontweight="bold")

for ax, attr, ylabel, color in zip(
    axes,
    ["separation", "coherence", "velocity"],
    ["Separation  S(l)", "Coherence  C(l)", "Velocity  v(l)"],
    [SIG_COLORS["separation"], SIG_COLORS["coherence"], SIG_COLORS["velocity"]],
):
    vals = [getattr(m, attr) for m in metrics]
    ax.plot(depth_pct, vals, color=color, lw=2)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.axhline(0, color=THEME["spine"], lw=0.8, ls="--")
    apply_theme(ax)
    for region in profile.regions:
        ax.axvline(region.depth_pct, color="#C62828", lw=1.5, ls=":", alpha=0.7)

axes[-1].set_xlabel("Depth (% of layers)", fontsize=11)
plt.tight_layout()
plt.show()

## 6. How does your concept compare?

The following cell downloads the pre-computed CAZ results for Qwen2.5-7B-Instruct across all 17 paper concepts and plots where your concept's peak lands on the shallow-to-deep ordering.

In [ ]:
# Baseline is always Qwen2.5-7B (the paper reference model).
# If you ran a different model the peak depths may differ slightly — that is
# expected and scientifically interesting.  The 7B baseline gives a fixed
# reference frame to see where your concept sits in concept-space.
HF_REPO = "james-ra-henry/Rosetta-Activations"
BASELINE_MODEL_KEY = "Qwen_Qwen2.5_7B"

PAPER_CONCEPTS = [
    "agency", "authorization", "causation", "certainty", "credibility",
    "deception", "exfiltration", "formality", "moral_valence", "negation",
    "plurality", "sarcasm", "sentiment", "specificity", "temporal_order",
    "threat_severity", "urgency",
]

baseline = {}
for c in PAPER_CONCEPTS:
    path = hf_hub_download(
        HF_REPO,
        filename=f"models/{BASELINE_MODEL_KEY}/caz_{c}.json",
        repo_type="dataset",
    )
    with open(path) as f:
        baseline[c] = json.load(f)["layer_data"]["peak_depth_pct"]

your_peak = profile.dominant.depth_pct

by_depth = sorted(baseline.items(), key=lambda x: x[1])
names = [c for c, _ in by_depth]
depths = [d for _, d in by_depth]

fig, ax = plt.subplots(figsize=(11, 4.5))
fig.patch.set_facecolor("white")

bar_colors = [concept_color(c) for c in names]
ax.barh(names, depths, color=bar_colors, height=0.65, alpha=0.85)

if CONCEPT_NAME not in baseline:
    ax.axvline(your_peak, color="#C62828", lw=2.5, ls="--",
               label=f"'{CONCEPT_NAME}' — your run ({your_peak:.1f}%)")
    ax.legend(fontsize=10)

ax.set_xlabel("Peak depth (% of layers)", fontsize=11)
_model_note = f"Qwen2.5-7B baseline" if MODEL_ID != "Qwen/Qwen2.5-7B" else MODEL_ID.split('/')[-1]
ax.set_title(
    f"Peak depth: your concept vs. 17 paper concepts  ({_model_note})",
    fontsize=12, fontweight="bold",
)
apply_theme(ax)
plt.tight_layout()
plt.show()

if CONCEPT_NAME in baseline:
    print(f"'{CONCEPT_NAME}' is a paper concept.")
    print(f"  Paper value (Qwen2.5-7B):  {baseline[CONCEPT_NAME]:.1f}%")
    print(f"  Your run ({MODEL_ID.split('/')[-1]}):   {your_peak:.1f}%")
    if MODEL_ID != "Qwen/Qwen2.5-7B":
        print("  Note: running a different model — depth differences are expected.")
else:
    rank = sum(1 for d in depths if d < your_peak) + 1
    print(f"'{CONCEPT_NAME}' peaks at {your_peak:.1f}%  —  rank {rank} of {len(depths)+1} (shallow → deep)")
    if MODEL_ID != "Qwen/Qwen2.5-7B":
        print(f"  (Baseline is Qwen2.5-7B; absolute depths will differ on {MODEL_ID.split('/')[-1]}.)")

## Next steps

- Try different concepts — does *uncertainty* behave differently from *certainty*? Does *irony* look like *sarcasm*?
- Swap `MODEL_ID` for any HuggingFace model: `extract_contrastive_activations` works with any model that exposes `output_hidden_states=True`
- **`03_caz_implementation_demo.ipynb`** — implement the metrics and Procrustes alignment from scratch, and reproduce the cross-architecture convergence result from Paper 4

| Paper | |
|-------|-|
| Paper 1 — CAZ Framework | [Henry 2026a](https://arxiv.org/abs/PLACEHOLDER) |
| Paper 4 — Cross-architecture PRH | [Henry 2026d](https://arxiv.org/abs/PLACEHOLDER) |
| `rosetta_tools` | [GitHub](https://github.com/jamesrahenry/Rosetta_Tools) |